<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Zenodo B23 / Solar Fusion III — Online Data EDA

This notebook reads the provider's public online resources directly, without using the converted TPeanuts tables. It inventories the remote content, checks formats and units, and determines which canonical products can be generated.

Official source: [https://zenodo.org/records/10822316](https://zenodo.org/records/10822316)

## Available products

- Density and composition: available for six solar compositions.
- Production: radial distributions for eight neutrino sources.
- Flux: totals, uncertainties and correlation matrices.
- Energy spectrum and survival probability: not supplied.

## 1. Imports and remote access

In [1]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Online inventory and format inspection

In [2]:
RECORD_API = "https://zenodo.org/api/records/10822316"
record = json.loads(fetch(RECORD_API))
inventory = pd.DataFrame([{"key": item["key"], "size": item["size"], "checksum": item["checksum"], "url": item["links"]["self"]} for item in record["files"]])
display(inventory)
archive = fetch(record["files"][0]["links"]["self"])
with zipfile.ZipFile(BytesIO(archive)) as bundle:
    members = pd.DataFrame({"member": bundle.namelist(), "bytes": [bundle.getinfo(name).file_size for name in bundle.namelist()]})
display(members.head(30))

,key,size,checksum,url
0,SolarModels_ICE2023_v1.2.zip,2510334,md5:2a3725d625bb1ee24aa39b82140a3062,https://zenodo.org/api/records/10822316/files/...


,member,bytes
0,SolarModels/,0
1,SolarModels/Cs_obs.dat,2652
2,SolarModels/fluxes_SF3_AAG21.dat,1112
3,SolarModels/fluxes_SF3_AGSS09.dat,1104
4,SolarModels/fluxes_SF3_C11.dat,1095
5,SolarModels/fluxes_SF3_GS98.dat,1099
6,SolarModels/fluxes_SF3_MB22m.dat,1103
7,SolarModels/fluxes_SF3_MB22p.dat,1116
8,SolarModels/struct+nu_SF3_AAG21.dat,1515506
9,SolarModels/struct+nu_SF3_AGSS09.dat,1515506


## 3. Conclusions

Only source-published observables are attributed to this provider. Derived TPeanuts quantities and detector-level spectra are labelled separately by the generator notebook.